# task 3

In [ ]:
from ase import Atoms
import numpy as np
from ase import Atoms
from ase.io import read, write
from ase.optimize import BFGS
from gpaw import GPAW, FermiDirac

positions = np.array([
    [0.0, 0.0, 0.0],
    [2.5, 0.0, 0.0],
    [0.0, 2.5, 0.0],
    [2.5, 2.5, 0.0],
    [1.25, 1.25, 2.0],
    [1.25, 2.0, 1.25]
])
atoms = Atoms('Na6', positions=positions)

atoms.center(vacuum=5.0) # 5 Å vacuum in x, y, and z directions


calc = GPAW(
    mode='lcao',              # linear combination of atomic orbitals
    basis='dzp',              # double-zeta polarized basis set
    xc='PBE',                 # planewave basis set
    occupations=FermiDirac(0.1),
    txt='calc.log'            
)

atoms.calc = calc

# relax the structure
dyn = BFGS(atoms, logfile='opt.log')
dyn.run(fmax=0.05, steps=200)  # terminate when the forces are less than 0.05 eV/Å or 200 steps

# final energy
energy = atoms.get_potential_energy()
print(f"Final energy (eV) = {energy:.6f}")

calc.write('final_guess.gpw', mode='all')

write('final_guess.traj', atoms)


# task 5

In [ ]:
from ase.db import connect
from ase.io import write

# Connect to the database 
db = connect('./A2/na8_gadb.db') # ATTENTION: changing for Na8/Na7/Na6/

# Find the structure with the lowest energy
rows = db.select(sort='energy', limit=1)
lowest_energy_row = next(rows)
avg_atom_energy  =  lowest_energy_row.energy / 8 # ATTENTION: changing for Na8/Na7/Na6/
lowest_energy_structure = lowest_energy_row.toatoms()

print(f"Atoms average lowest energy: {avg_atom_energy:.6e} eV")
write('Na8_lowest_energy_structure.xyz', lowest_energy_structure) # ATTENTION: changing for Na8/Na7/Na6/



# task 7

In [ ]:
from ase.db import connect
from ase.io import write
import numpy as np
import matplotlib.pyplot as plt


db = connect('./A2/na8_gadb.db') # ATTENTION: changing for Na8/Na7/Na6/
energies = []
indices = []
rows = db.select()

for idx, row in enumerate(rows):
    atoms = row.toatoms()
    if atoms.calc is not None:
        energies.append(atoms.get_total_energy())
    else:
        energies.append(np.nan)
    indices.append(idx)

valid_indices = [idx for idx, e in zip(indices, energies) if not np.isnan(e)]
valid_energies = [e for e in energies if not np.isnan(e)]

if valid_energies:  
    min_energy = min(valid_energies)
    min_index = valid_indices[valid_energies.index(min_energy)]
    print(f"Lowest energy: {min_energy:.6e} eV")
else:
    min_energy, min_index = None, None

plt.figure(figsize=(10, 6))
sc = plt.scatter(indices, energies, c=energies, cmap='viridis', edgecolors='k', s=50) 
plt.colorbar(label="Energy (eV)")

if min_energy is not None:
    plt.scatter(min_index, min_energy, color='red', s=100, marker='*', label=f"Lowest Energy ({min_energy:.4f} eV)")
    plt.legend()


for i, (x, y) in enumerate(zip(indices, energies)):
    plt.text(x, y, str(i), fontsize=8, ha='right', va='bottom')


plt.xlabel("Structure Index")
plt.ylabel("Energy (eV)")
plt.title("Task7 Na8 Energy Distribution of Structures") # ATTENTION: changing for Na8/Na7/Na6/
plt.legend()
plt.grid(True)
plt.savefig('./A2/task7_na8_energy_distribution.png') # ATTENTION: changing for Na8/Na7/Na6/
plt.show()

id_second_lowest = 75
print(f"length of energies: {len(energies)}")

second_lowest_energy = energies[id_second_lowest]  # eV
print(f"Second lowest energy: {second_lowest_energy:.6e} eV")
second_lowest_structure = db.get_atoms(id=id_second_lowest)
write(f'./A2/task7_Na8_structure_{id_second_lowest}.xyz', second_lowest_structure) # ATTENTION: changing for Na8/Na7/Na6/


# task 8

In [ ]:
import copy
from ase.io import read, write
from gpaw import GPAW, PW
from ase.optimize import BFGS

# Read standard structures
struct1 = read('standard_christmas_tree.xyz')
struct2 = read('standard_half_decahedron.xyz')

# Define at least 6 different parameter sets
parameter_sets = [
    {"mode": PW(350), "xc": "PBE"},          # Standard PW
    #{"mode": PW(400), "xc": "PBE"},   
    #{"mode": PW(500), "xc": "PBE"},        # Higher cutoff PW
    #{"mode": 'fd', "h": 0.2, "xc": "PBE"},   # Standard fd
    #{"mode": 'fd', "h": 0.15, "xc": "PBE"},  # Finer grid fd
    #{"mode": PW(350), "xc": "LDA"}  
    #{"mode": PW(400), "xc": "LDA"}
    #{"mode": 'lcao', "basis": "dzp", "xc": "PBE"},  # Double-zeta polarized
    # {"mode": 'lcao', "basis": "dzp", "xc": "LDA"}   
    #{"mode": 'lcao', "basis": "tzp", "xc": "PBE"},
    #{"mode": 'lcao', "basis": "qzp", "xc": "PBE"},
    #{"mode": PW(400), "xc": "HSE06"},
    #{"mode": PW(400), "xc": "PBE0"},
    #{"mode": PW(400), "xc": "SCAN"},

]

results = []

# Run calculations for both structures with all parameter sets
for struct_id, structure in enumerate([struct1, struct2], 1):
    for i, params in enumerate(parameter_sets):
        print(f"Running calculation for structure {struct_id} with params: {params}")
        calc_params = params.copy()
        mode = calc_params.pop("mode")

        calc = GPAW(
            mode=mode, 
            #h=params["h"],
            xc=params["xc"],
            #nbands=-10, 
            #kpts=(1, 1, 1),
            txt=f'task8_struct{struct_id}_calc_{i+1}.txt',
            #parallel={'domain': 1}
        )
        #structure.set_calculator(calc)
        structure.calc = calc
        opt = BFGS(structure)
        opt.run(fmax=0.01)
        energy = structure.get_potential_energy()
        results.append((struct_id, params, energy))
        write(f'task8_struct{struct_id}_relaxed_{i+1}.xyz', structure)

print("\nResults Table:")
print("{:<10} | {:<40} | {:<15}".format("Structure", "Parameters", "Energy (eV)"))
print("-" * 70)
for struct_id, params, energy in results:
    # Format parameters string based on mode type
    mode = params['mode']
    if isinstance(mode, PW):
        params_str = f"PW({mode.ecut}) eV, {params['xc']}"
    elif params['mode'] == 'fd':
        params_str = f"FD(h={params['h']}), {params['xc']}"
    elif params['mode'] == 'lcao':
        params_str = f"LCAO({params['basis']}), {params['xc']}"
    else:
        params_str = str(params)
        
    print("{:<10} | {:<40} | {:<15.6f}".format(f"Struct {struct_id}", params_str, energy))

# task 10

In [ ]:
from ase.io import read, write
from gpaw import GPAW, PW, restart
from ase.units import Bohr
import numpy as np

# Define the clusters
clusters = {
    'Na6': read('./A2/Na6_lowest_energy_structure.xyz'),
    'Na7': read('./A2/Na7_lowest_energy_structure.xyz'),
    'Na8': read('./A2/Na8_lowest_energy_structure.xyz')
}

# Perform GPAW calculations and save wavefunctions
for name, cluster in clusters.items():
    calc = GPAW(mode = PW(450), 
                xc='PBE', txt=f'task10_{name}_gpaw_output.log',
                nbands=28, 
                spinpol=False, 
                symmetry='off', 
                setups={'Na': '1'}
                )

    cluster.calc = calc  
    energy = cluster.get_potential_energy()
    calc.write(f'./A2/task10_{name}_wavefunctions.gpw', mode='all')  # Save all data including wavefunctions
    print(f'{name} energy: {energy} eV')

# Save wavefunctions to cube files
for name in clusters.keys():
    atoms, calc = restart(f'./A2/task10_{name}_wavefunctions.gpw')
    nbands = calc.get_number_of_bands()
    nelectrons = calc.get_number_of_electrons()
    occupied_bands = int(np.ceil(nelectrons / 2))
    print(f'{name} total bands: {nbands}')
    print(f'{name} occupied bands: {occupied_bands}')

    for band in range(occupied_bands):
        wf = calc.get_pseudo_wave_function(band=band)
        fname = f'./A2/task10_{name}_band_{band}.cube'
        print(f'writing wf {band} to file {fname}')
        write(fname, atoms, data=wf * Bohr**1.5)